# Tutorial: Train and validate models
### 1. Load data and related packages
### 2. Train various models
#### 2.1 Train and tune LightGBM
#### 2.2 Visualize the results
#### 2.3 Save and load your Trainer
#### 2.4 Train other models
### 3. Cross validate models

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Load data and related packages

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

UCI Adult data are going to be used in this tutorial. The data contain both numerical and categorical features, and the target to predict is whether a person makes over 50K a year. It is a very simple binary classification task.

In [ ]:
train = pd.read_csv('kuma_utils/datasets/adult_train.csv')
test = pd.read_csv('kuma_utils/datasets/adult_test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.info()

Let's prepare the data. We need to encode categorical features.

In [ ]:
from xfeat import SelectCategorical, SelectNumerical, LabelEncoder
from kuma_utils.preprocessing.xfeat import Pipeline, TargetEncoder

In [ ]:
num_enc = Pipeline([
    SelectNumerical(),
])

cat_enc = Pipeline([
    SelectCategorical(),
    LabelEncoder(output_suffix='')
], target_col='Target')

In [ ]:
train_cat, train_num = cat_enc.fit_transform(train), num_enc.fit_transform(train)
categorical_index = list(range(train_cat.shape[1]))
train_encoded = pd.concat([train_cat, train_num], axis=1) 
train_encoded.head()

Looks good to me. Let's apply the same transform to test data.

In [ ]:
test_encoded = pd.concat([cat_enc.transform(test), num_enc.transform(test)], axis=1)
test_encoded.head()

In [ ]:
labels, features = train_encoded['Target'], train_encoded.drop('Target', axis=1)
labels_test, features_test = test_encoded['Target'], test_encoded.drop('Target', axis=1)

## 2. Train models
Before going to training, let's define our evaluation metric. Since the task is binary classification, I will use ROC-AUC for evaluation.

In [ ]:
from sklearn.metrics import roc_auc_score, make_scorer
from kuma_utils.training import Trainer, auc_metric, acc_metric

### 2.1 Use Trainer

In [ ]:
x_train, x_valid, y_train, y_valid = train_test_split(features, labels, test_size=0.2)

In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1
}
fit_params = {
    'num_boost_round': 500,
    'early_stopping_rounds': 50,
    'verbose_eval': 25,
}

trn = Trainer(LGBMClassifier)
trn.train(
    train_data=(x_train, y_train), 
    valid_data=(x_valid, y_valid),
    cat_features=categorical_index,
    # logger='path_to_log', 
    params=params, fit_params=fit_params, 
)

In [ ]:
trn

In [ ]:
roc_auc_score(labels_test, trn.predict_proba(features_test)[:, 1])

Also, Trainer class has a built-in optuna function. Just set `tune_model=True` and it will search the best hyperparameters. Don't forget to set `n_trials` or `timeout` to prevent endless tuning (For LightGBM models, tuning will finished by itself).

In [ ]:
trn = Trainer(LGBMClassifier)
trn.train(
    train_data=(x_train, y_train), 
    valid_data=(x_valid, y_valid),
    cat_features=categorical_index,
    params=params, fit_params=fit_params, 
    tune_model=True
)

In [ ]:
roc_auc_score(labels_test, trn.predict_proba(features_test)[:, 1])

In [ ]:
trn

Custom metrics are available.

In [ ]:
from kuma_utils.metrics import SeWithFixedSp

params = {
    'objective': 'binary',
    'metric': 'SeWithFixedSp',
    'verbose': -1
}
fit_params = {
    'num_boost_round': 500,
    'early_stopping_rounds': 50,
    'verbose_eval': 25,
    'feval': SeWithFixedSp(sp=0.9).lgbm
}

trn = Trainer(LGBMClassifier)
trn.train(
    train_data=(x_train, y_train), 
    valid_data=(x_valid, y_valid),
    cat_features=categorical_index,
    # logger='path_to_log', 
    params=params, fit_params=fit_params, 
)

### 2.2 Visualize the results

In [ ]:
trn.plot_feature_importance()
trn.plot_feature_importance('gain')
trn.plot_feature_importance(
    'permutation', fit_params={
        'X': x_valid, 'y': y_valid, 'scoring': make_scorer(roc_auc_score), 'n_jobs': -1})

In [ ]:
trn.plot_calibration_curve(data=(x_train, y_train))

### 2.3 Save and load Trainer

In [ ]:
trn.save('lgbm0.trainer')
trn = Trainer(path='lgbm0.trainer')
trn

### 2.4 Train other models

In [ ]:
params = {
    'objective': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 500,
    'use_best_model': True,
} 
fit_params = {
    'early_stopping_rounds': 50,
    'verbose_eval': 25
}

trn2 = Trainer(CatBoostClassifier)
trn2.train(
    train_data=(x_train, y_train), 
    valid_data=(x_valid, y_valid),
    cat_features=categorical_index,
    params=params,
    tune_model=True, n_trials=5,
    fit_params=fit_params)

In [ ]:
roc_auc_score(labels_test, trn2.smart_predict(features_test))

In [ ]:
params = {}
fit_params = {}

trn3 = Trainer(LogisticRegression)
trn3.train(
    train_data=(x_train, y_train), 
    valid_data=(x_valid, y_valid),
    params=params,
    eval_metric=auc_metric, maximize=True,
    # tune_model=True, n_trials=5,
    fit_params=fit_params)

In [ ]:
roc_auc_score(labels_test, trn3.predict_proba(features_test)[:, 1])

## 3. Cross validate models

In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1
}
fit_params = {
    'num_boost_round': 500,
    'early_stopping_rounds': 50,
    'verbose_eval': 25,
}

trn = Trainer(LGBMClassifier)
trn.cv(
    data=(features, labels), 
    cat_features=categorical_index,
    folds=StratifiedKFold(n_splits=5), 
    params=params, fit_params=fit_params, 
)

In [ ]:
trn

In [ ]:
roc_auc_score(labels_test, np.stack(trn.smart_predict(features_test)).mean(0))

**Note that Trainer.cv() is cross validation during training process.** The model is fit to the whole data rather than each fold. This returns models with same hyperparameters, but trained on different data.

In [ ]:
from kuma_utils.training import CrossValidator

In [ ]:
validator = CrossValidator(LGBMClassifier)
validator.train(
    data=(features, labels), params=params, fit_params=fit_params,
    folds=StratifiedKFold(n_splits=5)
)

In [ ]:
validator

In [ ]:
roc_auc_score(labels_test, np.stack(validator.smart_predict(features_test)).mean(0))

In [ ]:
validator.plot_feature_importance(),
validator.plot_calibration_curve(data=(features, labels))

This one is an intuitive cross validation, in which models are fit to each fold.